In [1]:
!free -h
!nvidia-smi --query-gpu=name,memory.total --format=csv

               total        used        free      shared  buff/cache   available
Mem:            52Gi       1.4Gi        48Gi       2.2Mi       3.8Gi        51Gi
Swap:             0B          0B          0B
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


install and record versions

In [2]:
!pip install -q vllm==0.30.0 openai pillow requests

import os, io, sys, json, time, base64, subprocess, requests
import vllm, torch, transformers
from PIL import Image
from google.colab import userdata

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"

VERSIONS = {
    "python": sys.version.split()[0],
    "vllm": vllm.__version__,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "model": MODEL,
}
print(json.dumps(VERSIONS, indent=2))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

start the vLLM server

In [3]:
SERVER_ARGS = ["--max-model-len", "16384", "--gpu-memory-utilization", "0.85", "--port", "8000"]

log = open("/content/server_gate1.log", "w")
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, *SERVER_ARGS],
    stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())

t0 = time.time()
while True:
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print(f"Server up in {time.time() - t0:.0f}s"); break
    except requests.exceptions.RequestException:
        pass
    if server.poll() is not None or time.time() - t0 > 900:
        print("Server failed. Last log lines:")
        print(open("/content/server_gate1.log").read()[-3000:]); break
    time.sleep(5)

Server up in 295s


prepare the stop sign image

In [4]:
from vllm.assets.image import ImageAsset

img = ImageAsset("stop_sign").pil_image.convert("RGB")
buf = io.BytesIO()
img.save(buf, format="JPEG", quality=95)
JPEG_BYTES = buf.getvalue()

DATA_URL = "data:image/jpeg;base64," + base64.b64encode(JPEG_BYTES).decode()
SENT_IMG = Image.open(io.BytesIO(JPEG_BYTES)).convert("RGB")   # exactly what vLLM will decode
W, H = SENT_IMG.size
print(f"width {W}, height {H}, JPEG {len(JPEG_BYTES)/1024:.0f} KB, base64 {len(DATA_URL)/1024:.0f} KB")

width 1300, height 876, JPEG 248 KB, base64 330 KB


gate test #1

In [5]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
PROMPT = "Describe this image in one sentence."

def ask(messages):
    """Send one non-streaming chat request to vLLM.

    Use temperature=0 and max_tokens=60 (same as the original test #1).
    Returns (prompt_tokens, answer_text).
    """
    r = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0,
        max_tokens=60,
    )
    return r.usage.prompt_tokens, r.choices[0].message.content

img_messages = [{
    "role": "user",
    "content": [
        {"type": "image_url", "image_url": {"url": DATA_URL}},
        {"type": "text", "text": PROMPT},
    ],
}]

txt_messages = [{
    "role": "user",
    "content": PROMPT,
}]

img_prompt_tokens, answer = ask(img_messages)
txt_prompt_tokens, _ = ask(txt_messages)

measured = img_prompt_tokens - txt_prompt_tokens
expected = round(H / 28) * round(W / 28)

print("answer:", answer)
print(f"prompt tokens with image: {img_prompt_tokens}, text only: {txt_prompt_tokens}")
print(f"measured image tokens: {measured}, 28x28 rule: {expected}, gap: {measured - expected}")

answer: A stop sign is in front of a red and gold gate with Chinese characters on it.
prompt tokens with image: 1454, text only: 26
measured image tokens: 1428, 28x28 rule: 1426, gap: 2


settle the marker question

In [6]:
from transformers import AutoProcessor

proc = AutoProcessor.from_pretrained(MODEL)

def count_tokens_hf(pil_img, prompt):
    """Build the exact prompt Qwen sees for (image + prompt) and count tokens by type.

    Returns a dict: total, vision_start, image_pad, vision_end, grid.
    """
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": prompt}]}]

    text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    print(text)

    inputs = proc(text=[text], images=[pil_img], return_tensors="pt")
    ids = inputs["input_ids"][0].tolist()

    tid = proc.tokenizer.convert_tokens_to_ids
    return {
        "total": len(ids),
        "vision_start": ids.count(tid("<|vision_start|>")),
        "image_pad": ids.count(tid("<|image_pad|>")),
        "vision_end": ids.count(tid("<|vision_end|>")),
        "grid": inputs["image_grid_thw"][0].tolist(),
    }

hf = count_tokens_hf(SENT_IMG, PROMPT)
print(hf)
print("HF total == vLLM prompt_tokens:   ", hf["total"] == img_prompt_tokens)
print("image_pad == 28x28 rule:          ", hf["image_pad"] == expected)
print("start + pads + end == measured:   ", hf["vision_start"] + hf["image_pad"] + hf["vision_end"] == measured)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Describe this image in one sentence.<|im_end|>
<|im_start|>assistant

{'total': 1454, 'vision_start': 1, 'image_pad': 1426, 'vision_end': 1, 'grid': [1, 62, 92]}
HF total == vLLM prompt_tokens:    True
image_pad == 28x28 rule:           True
start + pads + end == measured:    True


save the result

In [7]:
from google.colab import drive
import datetime, shutil

drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/visionabr_results/phase_minus1"
os.makedirs(DEST, exist_ok=True)

result = {
    "test": "gate1_model_runs_and_token_count",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "versions": VERSIONS,
    "config": {
        "server_args": SERVER_ARGS,
        "image": "vllm ImageAsset stop_sign, JPEG quality 95, base64 data URL",
        "image_size_wh": [W, H],
        "prompt": PROMPT,
        "temperature": 0,
        "max_tokens": 60,
        "max_pixels": "default",
        "streaming": False,
    },
    "raw": {
        "answer": answer,
        "prompt_tokens_with_image": img_prompt_tokens,
        "prompt_tokens_text_only": txt_prompt_tokens,
        "hf_processor_counts": hf,
    },
    "summary": {
        "measured_image_tokens": measured,
        "rule_28x28": expected,
        "gap": measured - expected,
        "markers_verified": hf["vision_start"] == 1 and hf["vision_end"] == 1
                            and hf["vision_start"] + hf["image_pad"] + hf["vision_end"] == measured,
    },
}

stamp = datetime.date.today().isoformat()
path = f"{DEST}/{stamp}_gate1_tokens_stopsign.json"
json.dump(result, open(path, "w"), indent=2)
print("saved:", path)

shutil.copy("/content/server_gate1.log", f"{DEST}/{stamp}_gate1_server.log")
print("saved:", f"{DEST}/{stamp}_gate1_server.log")

Mounted at /content/drive
saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate1_tokens_stopsign.json
saved: /content/drive/MyDrive/visionabr_results/phase_minus1/2026-10-11_gate1_server.log
